<a href="https://colab.research.google.com/github/ancestor9/2026_Fall_Learning-Langchain-AI-Agent/blob/main/CHAPTER_3_RAG_Part_II_Chatting_with_Your_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -qU supabase langchain-community
!pip install -qU langchain-groq langchain-text-splitters langchain-core
!pip install -qU langchain-huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.6 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata
from supabase.client import create_client, Client
from langchain_community.vectorstores import SupabaseVectorStore
from langchain_huggingface import HuggingFaceEmbeddings # Updated import
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import chain

# ==========================================
# 1단계: 환경 변수 및 인증 정보 설정
# ==========================================
# 1) Supabase 접속 정보 (앞서 확인한 API 주소와 anon 키)
SUPABASE_URL = "https://tllrshlupjcikzaywaqm.supabase.co"
SUPABASE_KEY = userdata.get('supabase')

supabase_client: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

# 2) Groq API 키 설정
YOUR_API_KEY = userdata.get('groq')
os.environ["GROQ_API_KEY"] = YOUR_API_KEY

# ==========================================
# 2단계: 데이터 로드, 분할 및 임베딩 (Ingestion)
# ==========================================
# 문서 로드 및 분할
raw_documents = TextLoader('./test.txt', encoding='utf-8').load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(raw_documents)

# 수파베이스와 연동할 HuggingFace 384차원 임베딩 모델 설정
embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Supabase VectorStore에 데이터 Store (인덱싱 및 저장)
db = SupabaseVectorStore.from_documents(
    documents=documents,
    embedding=embeddings_model,
    client=supabase_client,
    table_name="documents",
    query_name="match_documents"  # SQL Editor에 등록한 함수 이름
)

# ==========================================
# 3단계: 검색기(Retriever) 및 프롬프트, Groq LLM 설정
# ==========================================
# 유사한(cosine similarity) 문서 2개를 뽑아오는 리트리버 생성
retriever = db.as_retriever(search_kwargs={"k": 2})

query = 'Who are the key figures in the ancient greek history of philosophy?'

# 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:
    {context}

    Question: {question}"""
)

# 초고속 가성비 모델인 Groq(Llama 3) 모델로 교체
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0)


# ==========================================
# 4단계: LCEL @chain 데코레이터를 활용한 효율적인 파이썬 함수 캡슐화 (Generation)
# ==========================================
print("Running RAG system using Groq and Supabase...\n")

@chain
def qa_chain(input_query):
    # 1. Retrieval: 질문과 관련된 문서 조각 가져오기
    docs = retriever.invoke(input_query)

    # 2. Prompt 조립: 질문과 컨텍스트 결합
    formatted = prompt.invoke({"context": docs, "question": input_query})

    # 3. Generation: Groq LLM을 통해 답변 생성
    answer = llm.invoke(formatted)
    return answer

# RAG 체인 실행 및 결과 출력
result = qa_chain.invoke(query)
print("[최종 답변 결과]")
print(result.content)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Running RAG system using Groq and Supabase...

[최종 답변 결과]
Unfortunately, the provided context does not mention specific key figures in ancient Greek philosophy. It only mentions that "Greek philosophers and scientists" laid the groundwork for diverse fields of study, but it does not provide any names or details about individual philosophers.


In [ ]:
query = 'Explain briefly about the ancient greek history of philosophy?'

result = qa_chain.invoke(query)
print("[최종 답변 결과]")
print(result.content)

[최종 답변 결과]
The ancient Greek philosophers and scientists laid the groundwork for diverse fields of study, promoting a culture of inquiry, reason, and intellectual rigor. Their contributions advanced contemporary understanding and provided enduring frameworks that continue to influence modern thought and practice.


In [ ]:
query = 'Who are the key figures in the ancient greek history of philosophy?'
result = qa_chain.invoke(query)
print("[최종 답변 결과]")
print(result.content)

[최종 답변 결과]
Unfortunately, the provided context does not mention specific key figures in ancient Greek philosophy. It only mentions that "Greek philosophers and scientists" laid the groundwork for diverse fields of study, but it does not provide any names or details about individual philosophers.


In [ ]:
docs = retriever.invoke(query)
for d in docs:
    print(d.page_content)
    print("---")

Greek ideas about ethics, governance, and the role of individuals in society continue to inform modern ethical theories, political systems, and educational curricula.
Conclusion
The philosophical and scientific endeavors of ancient Greece represent a monumental chapter in human intellectual history. Greek philosophers and scientists laid the essential groundwork for diverse fields of study, promoting a culture of inquiry, reason, and intellectual rigor. Their contributions not only advanced contemporary understanding but also provided enduring frameworks that continue to influence modern thought and practice. By exploring the rich legacy of Greek philosophy and science, we gain insight into the enduring quest for knowledge and the pursuit of wisdom that characterizes human civilization.
---
Chapter 6: Economy and Trade in Ancient Greece
---
Greek ideas about ethics, governance, and the role of individuals in society continue to inform modern ethical theories, political systems, and edu

In [ ]:
file_path = './test.txt'

found_sentences = []
with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        if 'Thales' in line:
            found_sentences.append(line.strip())

if found_sentences:
    print(f"Found {len(found_sentences)} sentence(s) containing 'Thales' in {file_path}:")
    for sentence in found_sentences:
        print(sentence)
else:
    print(f"No sentences containing 'Thales' found in {file_path}.")

Found 15 sentence(s) containing 'Thales' in ./test.txt:
Pre-Socratic Philosophers: Focused on natural phenomena and the origins of the cosmos, including Thales, Anaximander, and Heraclitus.
Thales of Miletus: Proposed that water was the fundamental substance (arche) underlying all matter.
Thales and Anaximander: Proposed early models of the solar system and celestial mechanics.
II. Thales was the first to offer a purely physical explanation of the
genius of Athens. Thales and his successors down to Democritus were
detractors of everything Hellenic. Thales and the rest, we are told,
Thales, of Miletus, an Ionian geometrician and astronomer, about whose
to Thales all things have come from water. That the earth is entirely
confused fancies of mythologising poets. Not that Thales was an
interferences with the course of Nature. For the rest, Thales made
and astronomy. We have seen that to Thales water, the all-embracing
were to them precisely what water had been to Thales, what air was to
r

In [ ]:
query = 'Who are Thales in the ancient greek history of philosophy?'
result = qa_chain.invoke(query)
print("[최종 답변 결과]")
print(result.content)

[최종 답변 결과]
Thales is mentioned in the provided context as one of the successors down to Democritus, who were not exactly what we should call philosophers in any sense of the word. However, it is implied that Thales and his successors were part of the early series of speculations on the nature of things in ancient Greek philosophy.


In [ ]:
query = input("질문을 입력하세요: ")
result = qa_chain.invoke(query)
print("[최종 답변 결과]")
print(result.content)

질문을 입력하세요: quit
[최종 답변 결과]
There is not enough information to answer your question.


### **왜 key features 를 못 찾지? 문서에는 있는데?**
- chunk_size=1000으로 쪼개면 chunk 수가 엄청 많아지게 됨 (대략 총 원문 9600줄 → 수백~수천 개 chunk).
- k=2는 그 수많은 chunk 중 딱 2개만 가져옵니다. 확률적으로 플라톤/아리스토텔레스 상세 내용이 담긴 chunk가 상위 2개 안에 들 가능성이 매우 낮음
- 실제로 검색된 2개는 아마 1장 초반부의 "Greek philosophers and scientists laid the groundwork..." 같은 일반적인 요약 문장일 듯 --> 답변 내용과 정확히 일치
- Supabase의 documents 테이블에는 약 780개의 벡터가 저장되어 있고, k=2는 이 780개 중 **딱 2개(전체의 0.25%)** 만 가져오는 것으로 질문과 의미적으로 가장 가까운 상위 2개가 하필 챕터 1 서두의 일반적인 요약 문장이었고, 철학자 이름이 담긴 chunk(97~300줄 근처)는 780개 중 3등, 5등, 10등 등 어딘가에 있었지만 k=2 밖으로 밀려나서 아예 전달되지 않은 것으로, LLM 입장에서는 "context에 없으니 모른다"고 정직하게 답한 것임
- **해결방안** ---> retriever = db.as_retriever(search_kwargs={"k": 10}) 로 코드를 변경하라

# **Query Transformation**

In [ ]:
# @title **Rewrite-Retrieve-Read**
# simply prompts the LLM to rewrite the user’s query before performing retrieval


# create retriever to retrieve 2 relevant documents
retriever = db.as_retriever(search_kwargs={"k": 3})

@chain
def qa(input_query):
    # 1. Retrieval: 질문과 관련된 문서 조각 가져오기(fetch relevant documents)
    docs = retriever.invoke(input_query)

    # 2. Prompt 조립: 질문과 컨텍스트 결합(format prompt)
    formatted = prompt.invoke({"context": docs, "question": input_query})

    # 3. Generation: Groq LLM을 통해 답변 생성(generate answer)
    answer = llm.invoke(formatted)
    return answer

result1 = qa.invoke("""Today I woke up and brushed my teeth, then I sat down to read the
 news. But then I forgot the food on the cooker. Who are some key figures in
 the ancient greek history of philosophy?""")

print(result1.content)

It seems like you're looking for information on ancient Greek philosophy, but the context you provided appears to be about the influence of ancient Greek ideas on modern thought and practice. However, I can still provide you with some key figures in ancient Greek philosophy.

Some of the most influential figures in ancient Greek philosophy include:

1. Socrates (469/470 BCE - 399 BCE): Known for his method of questioning, which is now called the Socratic method. He is considered one of the founders of Western philosophy.
2. Plato (428/427 BCE - 348/347 BCE): A student of Socrates, Plato founded the Academy in Athens, one of the earliest institutions of higher learning in the Western world. He wrote extensively on philosophy, politics, and metaphysics.
3. Aristotle (384 BCE - 322 BCE): A student of Plato, Aristotle made significant contributions to various fields, including philosophy, science, and ethics. He founded the Lyceum in Athens and wrote extensively on topics such as logic, me

In [ ]:
rewrite_prompt = ChatPromptTemplate.from_template("""Provide a better search
 query for web search engine to answer the given question, end the queries
 with ’**’. Question: {x} Answer:""")

def parse_rewriter_output(message):
    return message.content.strip('"').strip("**")

rewriter = rewrite_prompt | llm | parse_rewriter_output

@chain
def qa_rrr(input):
    # rewrite the query
    new_query = rewriter.invoke(input)
    # fetch relevant documents
    docs = retriever.invoke(input)
    # format prompt
    formatted = prompt.invoke({"context": docs, "question": input})
    # generate answer
    answer = llm.invoke(formatted)
    return answer

# run
result2 = qa_rrr.invoke("""Today I woke up and brushed my teeth, then I sat down to read
 the news. But then I forgot the food on the cooker. Who are some key
 figures in the ancient greek history of philosophy?""")


print(result2.content)

Unfortunately, the provided context does not mention anything about key figures in ancient Greek history of philosophy. It appears to be a repetition of the same text about the influence of ancient Greek ideas on modern thought and practice. However, based on general knowledge, some key figures in ancient Greek history of philosophy include:

1. Socrates: Known for his method of questioning, which is now called the Socratic method.
2. Plato: A student of Socrates, who founded the Academy in Athens and wrote many influential philosophical works.
3. Aristotle: A student of Plato, who made significant contributions to various fields, including philosophy, science, and ethics.
4. Epicurus: Founder of Epicureanism, a school of thought that emphasized the pursuit of happiness and the avoidance of pain.
5. Zeno of Citium: Founder of Stoicism, a school of thought that emphasized reason, self-control, and indifference to external events.

These are just a few examples, and there were many other

In [ ]:
# @title LCEL, No RAG
message_runnable = rewrite_prompt | llm
example_input = {"x": "Who are some key figures in ancient Greek philosophy?"}
response = message_runnable.invoke(example_input)
print(response.content)

Here are some improved search queries to find key figures in ancient Greek philosophy:

1. **"Ancient Greek philosophers list"**
2. **"Key figures in ancient Greek philosophy"**
3. **"Ancient Greek philosophers timeline"**
4. **"Notable ancient Greek philosophers"**
5. **"Ancient Greek philosophy influential thinkers"**
6. **"Ancient Greek philosophers contributions"**
7. **"Ancient Greek philosophy major figures"**
8. **"Ancient Greek philosophers and their ideas"**
9. **"Ancient Greek philosophy notable scholars"**
10. **"Ancient Greek philosophers and their impact"**

These search queries should provide a good starting point for finding information on key figures in ancient Greek philosophy.


In [ ]:
# @title **Multi-Query Retrieval**
# A user’s single query can be insufficient to capture the full scope of information required to answer the query comprehensively

perspectives_prompt = ChatPromptTemplate.from_template("""You are an AI language
 model assistant. Your task is to generate five different versions of the
 given user question to retrieve relevant documents from a vector database.
 By generating multiple perspectives on the user question, your goal is to
 help the user overcome some of the limitations of the distance-based
 similarity search. Provide these alternative questions separated by
 newlines. Original question: {question}""")

def parse_queries_output(message):
    return message.content.split('\n')

query_gen = perspectives_prompt | llm | parse_queries_output

In [ ]:
def get_unique_union(document_lists):
    # Flatten list of lists, and dedupe them
    deduped_docs = {
    doc.page_content: doc
    for sublist in document_lists for doc in sublist
    }
    # return a flat list of unique docs
    return list(deduped_docs.values())

retrieval_chain = query_gen | retriever.batch | get_unique_union

In [ ]:
prompt = ChatPromptTemplate.from_template("""Answer the following question based
 on this context:
{context}
Question: {question}
""")

@chain
def multi_query_qa(input):
    # fetch relevant documents
    docs = retrieval_chain.invoke(input)
    # format prompt
    formatted = prompt.invoke({"context": docs, "question": input})
    # generate answer
    answer = llm.invoke(formatted)
    return answer

# run
result3 = multi_query_qa.invoke("""Who are some key figures in the ancient greek history
 of philosophy?""")


print(result3.content)

Based on the provided context, some key figures in the ancient Greek history of philosophy mentioned are:

1. Thales of Miletus: He proposed that water was the fundamental substance (arche) underlying all matter.
2. Anaximander: He introduced the concept of the apeiron (infinite) as the origin of all things.

Additionally, the text mentions that the chapter will delve into the major philosophical schools, key scientific advancements, influential thinkers, and the enduring impact of Greek intellectual pursuits on subsequent generations. However, it does not provide a comprehensive list of key figures in ancient Greek philosophy.


In [ ]:
# @title **RAG-Fusion**
#The RAG-Fusion strategy shares similarities with the multi-query retrieval strategy, except we will apply a final reranking step to all the retrieved documents.

In [ ]:
prompt_rag_fusion = ChatPromptTemplate.from_template("""You are a helpful
 assistant that generates multiple search queries based on a single input
 query. \n
 Generate multiple search queries related to: {question} \n
 Output (4 queries):""")

def parse_queries_output(message):
    return message.content.split('\n')

generate_queries = prompt_rag_fusion | llm | parse_queries_output

In [ ]:
from typing import List

def reciprocal_rank_fusion(results: List[List], k=60):
    """reciprocal rank fusion on multiple lists of ranked documents
    and an optional parameter k used in the RRF formula
    """

    # Initialize a dictionary to hold fused scores for each document
    # Documents will be keyed by their contents to ensure uniqueness
    fused_scores = {}
    documents = {}

    # Iterate through each list of ranked documents
    for docs in results:
        # Iterate through each document in the list,
        # with its rank (position in the list)
        for rank, doc in enumerate(docs):
            # Use the document contents as the key for uniqueness
            doc_str = doc.page_content

            # If the document hasn't been seen yet,
            # - initialize score to 0
            # - save it for later
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
                documents[doc_str] = doc

            # Update the score of the document using the RRF formula:
            # 1 / (rank + k)
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort the documents based on their fused scores in descending order
    # to get the final reranked results
    reranked_doc_strs = sorted(
        fused_scores, key=lambda d: fused_scores[d], reverse=True
    )

    # retrieve the corresponding doc for each doc_str
    return [
        documents[doc_str]
        for doc_str in reranked_doc_strs
    ]

# LangChain Expression Language (LCEL) Chain 구성
retrieval_chain = generate_queries | retriever.batch | reciprocal_rank_fusion

In [ ]:
# 1. 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_template("""Answer the following question based
 on this context:
{context}
Question: {question}
""")

# 2. @chain 데코레이터를 활용한 전체 QA 파이프라인 구성
@chain
def multi_query_qa(input_text):
    # RAG-Fusion 체인을 통해 재정렬된 관련 문서들을 가져옵니다.
    docs = retrieval_chain.invoke(input_text)

    # 가져온 문서(context)와 질문(question)을 프롬프트에 주입합니다.
    formatted = prompt.invoke({"context": docs, "question": input_text})

    # LLM을 호출하여 최종 답변을 생성합니다.
    answer = llm.invoke(formatted)
    return answer

# 3. 체인 실행 (그리스 철학사 관련 질문)
# * 주의: 이 코드가 정상 작동하려면 이전 단계에서 정의한 `retrieval_chain`이
#   먼저 메모리에 선언되어 있어야 합니다.
result = multi_query_qa.invoke("Who are some key figures in the ancient greek history of philosophy?")

# 결과 출력
print(result.content)

Based on the provided context, some key figures in the ancient Greek history of philosophy are:

1. Aristotle: Known for his contributions to logic, metaphysics, ethics, politics, biology, and more.
2. Epicurus: Famous for his teachings on pleasure, happiness, and the nature of the universe, which influenced subsequent philosophical thought.
3. Zeno of Citium: The founder of Stoicism, who introduced ideas about rationality, emotional resilience, and living in accordance with nature.
4. Thales of Miletus: A Pre-Socratic philosopher who proposed that water was the fundamental substance (arche) underlying all matter.
5. Anaximander: A Pre-Socratic philosopher who introduced the concept of the apeiron (infinite) as the origin of all things.
6. Socrates: A prominent figure in Greek philosophy, known for his universal celebrity and his contributions to the development of Western philosophy.

These individuals played significant roles in shaping the foundations of Western philosophy and conti

In [ ]:
# @title **Hypothetical Document Embeddings**
# A strategy that involves creating a hypothetical document based on the user’s query,
# embedding the document, and retrieving relevant documents based on vector similarity.

from langchain_core.output_parsers import StrOutputParser

prompt_hyde = ChatPromptTemplate.from_template("""Please write a passage to
answer the question.\n Question: {question} \n Passage:""")

generate_doc = (
 prompt_hyde | llm | StrOutputParser()
)

In [ ]:
# 사용자의 질문이 아니라 LLM이 쓴 답변 문장을 벡터로 임베딩하여 실제 정답 문서들을 검색
 # 🔴 연빨간색 원을 통해 🟢 초록색/🔵 파란색 원을 찾아가는 과정

retrieval_chain = generate_doc | retriever

In [ ]:
prompt = ChatPromptTemplate.from_template("""Answer the following question based
 on this context:
{context}
Question: {question}
""")

@chain
def qa(input):
    # fetch relevant documents from the hyde retrieval chain defined earlier
    docs = retrieval_chain.invoke(input)
    # format prompt
    formatted = prompt.invoke({"context": docs, "question": input})
    # generate answer
    answer = llm.invoke(formatted)
    return answer

result4 = qa.invoke("""Who are some key figures in the ancient greek history of
 philosophy?""")

print(result4.content)

Based on the provided context, some key figures in the ancient Greek history of philosophy are:

1. **Socrates**: A philosopher who left no written records, but his ideas were recorded by his students, notably Plato.
2. **Plato**: A student of Socrates, who founded the Academy and developed a comprehensive philosophical system. He is known for his Theory of Forms, Political Philosophy, and Dialectic Method.
3. **Aristotle**: A student of Plato, who established the Lyceum and made significant contributions across multiple disciplines, including Empirical Observation, Logic, and Syllogism.

These three philosophers are considered some of the most influential figures in the history of Western philosophy, and their ideas continue to shape philosophical thought to this day.
